In [1]:
import joblib
import re
import nltk
import numpy as np
import gradio as gr
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from gensim.models import FastText

In [2]:
# # 1. Inisialisasi Preprocessing (Identik dengan tahap training)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [ps.stem(w) for w in text.split() if w not in stop_words]
    return tokens

def doc_vector(tokens, model_wv, vector_size=100):
    vecs = [model_wv[w] for w in tokens if w in model_wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)

In [3]:
# 2. Load Model & Setup Vectorizer
model_path = '../Model/best_fake_news_model.joblib'
model_ft_path = '../Model/best_fake_news_model_ft.bin'

print("⏳ Memuat model dan vectorizer...")
model = joblib.load(model_path)

# Load model FastText yang sudah disimpan
ft_model = FastText.load(model_ft_path)

print("✅ Sistem Siap!")

⏳ Memuat model dan vectorizer...
✅ Sistem Siap!


In [8]:
# 3. Fungsi Prediksi untuk Gradio
def predict_fake_news(text):
    if not text.strip(): return "Teks kosong."
    tokens = preprocess_text(text)
    vector = doc_vector(tokens, ft_model.wv, 839).reshape(1, -1)
    
    prediction = model.predict(vector)[0]
    prob = model.predict_proba(vector)[0]
    label = "REAL" if prediction == 0 else "FAKE"
    return f"Hasil: {label}"

In [9]:
# 4. Antarmuka Gradio
demo = gr.Interface(
    fn=predict_fake_news,
    inputs=gr.Textbox(lines=5, label="Masukkan Berita (English)"),
    outputs=gr.Textbox(label="Hasil Analisis"),
    title="Fake News Detector",
    description="Prediksi berita asli atau palsu menggunakan model terbaik (XGBoost + FastText).",
    flagging_mode="never"
)

demo.launch()

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.
